In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from peft import get_peft_model, LoraConfig
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

# --- Setup ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base")
base_model = AutoModel.from_pretrained(
    "deepseek-ai/deepseek-coder-6.7b-base",
    quantization_config=bnb_config,
    output_hidden_states=True,
    device_map={"": "cuda:0"}
)

# LoRA auf Attention-Layer
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)
base_model = get_peft_model(base_model, lora_config)
base_model.print_trainable_parameters()
# z.B.: trainable params: 4,194,304 || all params: 6,742,609,920

model = ExSLModel(base_model, hidden_size=4096).to("cuda")

# --- Training Loop ---
optimizer = AdamW(model.parameters(), lr=5e-6, weight_decay=0.0)
loss_fn   = nn.BCEWithLogitsLoss()

GRAD_ACCUM_STEPS = 16  # simuliert Batch Size 16
MAX_TOKENS       = 1024

model.train()
optimizer.zero_grad()

for step, item in enumerate(dataset['train']):
    sample = build_training_sample(item)
    prompt = sample['prompt']
    labels = torch.tensor(sample['labels'], dtype=torch.float16).to("cuda")

    # Tokenisieren (mit Token-Limit)
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_TOKENS,
        truncation=True
    ).to("cuda")

    # Marker-Positionen finden
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    open_pos  = [i for i, t in enumerate(tokens) if t == 'Â«']
    close_pos = [i for i, t in enumerate(tokens) if t == 'Â»']

    # Kandidaten die durch Truncation weggefallen sind überspringen
    n = min(len(open_pos), len(close_pos), len(labels))
    if n == 0:
        continue

    labels = labels[:n]

    # Forward + Loss
    logits = model(enc['input_ids'], enc['attention_mask'], open_pos[:n], close_pos[:n])
    loss   = loss_fn(logits, labels) / GRAD_ACCUM_STEPS
    loss.backward()

    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        optimizer.step()
        optimizer.zero_grad()
        print(f"Step {step+1:4d} | Loss: {loss.item() * GRAD_ACCUM_STEPS:.4f}")

# --- Modell speichern ---
model.base.save_pretrained("exsl_lora")
torch.save(model.w_relevance.state_dict(), "exsl_head.pt")